In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp7ejyjjme".


In [2]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>
#include <math.h> // Standard math library works in CUDA kernels

// THE KERNEL
__global__ void squareArray(float* data, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;

    if (i < n) {
        // You can use data[i] * data[i] OR the intrinsic function powf()
        float val = data[i];
        data[i] = val * val;
    }
}

int main() {
    int n = 10; // Keeping it small to see the full output
    size_t size = n * sizeof(float);

    float *h_data = (float*)malloc(size);
    // Initialize with numbers 0 to 9
    for(int i=0; i<n; i++) h_data[i] = (float)i;

    float *d_data;
    cudaMalloc(&d_data, size);
    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);

    // Launch: Using 256 threads per block (standard practice)
    squareArray<<<1, 256>>>(d_data, n);

    cudaMemcpy(h_data, d_data, size, cudaMemcpyDeviceToHost);

    // Print all results for verification
    std::cout << "Input index i -> Output i^2" << std::endl;
    std::cout << "---------------------------" << std::endl;
    for(int i=0; i<n; i++) {
        std::cout << i << " -> " << h_data[i] << std::endl;
    }

    cudaFree(d_data);
    free(h_data);
    return 0;
}

Input index i -> Output i^2
---------------------------
0 -> 0
1 -> 1
2 -> 4
3 -> 9
4 -> 16
5 -> 25
6 -> 36
7 -> 49
8 -> 64
9 -> 81

